# SpendSense AI: AI-powered Personal Finance Leakage Detection for Corporate Professionals

**Objective:** Build an end-to-end Data Analytics project focused on helping salaried corporate professionals (earning ₹35,000 – ₹1,50,000 per month) identify hidden financial leakages in their monthly expenses.

> **Disclaimer:** *The dataset used in this project is synthetically generated for educational and portfolio presentation purposes to represent realistic spending habits of corporate professionals. No actual personal data is included.*

## 1. Project Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# Set style for visualizations
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print("Setup completed successfully!")

## 2. Data Loading & Understanding

In [2]:
raw_path = "../data/raw/raw_transaction_data.csv"
df = pd.read_csv(raw_path)
print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 rows of the dataset:")
df.head()

In [3]:
print("Dataset Column Info:")
df.info()

In [4]:
print("Check unique values in categorical variables:")
for col in ['Salary_Band', 'Work_Mode', 'Transaction_Type', 'Category', 'Payment_Method', 'Expense_Type', 'Subscription']:
    print(f"{col}: {df[col].unique()}")

## 3. Data Cleaning

In this section, we verify the data integrity by checking for duplicates and missing values, and formatting the transaction date column.

In [5]:
# Convert date to datetime object
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df['Day_of_Week'] = df['Date'].dt.day_name()

# Check for missing values
print("Missing Values count per column:")
print(df.isnull().sum())

# Check for duplicate records
print(f"\nDuplicate rows found: {df.duplicated().sum()}")

## 4. Feature Engineering

We derive practical, business-relevant features from the raw transaction data:
1. **Weekend_Flag**: Transaction occurred on Saturday or Sunday.
2. **Salary_Week**: Transaction occurred within the first week after salary credit (Day 1 to 7).
3. **Monthly Savings & Savings %**: Aggregated monthly savings per corporate professional.
4. **is_leakage (Target)**: A binary label indicating potential expense leakage synthesized using rule-based scoring (only for wants).

In [6]:
# 1. Weekend Flag
df['Weekend_Flag'] = df['Date'].dt.weekday.apply(lambda x: 'Yes' if x >= 5 else 'No')

# 2. Salary Week Flag (Day 1 to 7 of month)
df['Salary_Week'] = df['Date'].dt.day.apply(lambda x: 'Yes' if x <= 7 else 'No')

# 3. Monthly Income, Expenses, and Savings per Employee
income_df = df[df['Transaction_Type'] == 'Income'].groupby(['Employee_ID', 'Year', 'Month'])['Amount'].sum().reset_index()
income_df.rename(columns={'Amount': 'Monthly_Income'}, inplace=True)

expense_df = df[df['Transaction_Type'] == 'Expense'].groupby(['Employee_ID', 'Year', 'Month'])['Amount'].sum().reset_index()
expense_df.rename(columns={'Amount': 'Monthly_Expense'}, inplace=True)

invest_df = df[df['Transaction_Type'] == 'Investment'].groupby(['Employee_ID', 'Year', 'Month'])['Amount'].sum().reset_index()
invest_df.rename(columns={'Amount': 'Monthly_Investment'}, inplace=True)

# Merge together
monthly_summary = pd.merge(income_df, expense_df, on=['Employee_ID', 'Year', 'Month'], how='left').fillna(0)
monthly_summary = pd.merge(monthly_summary, invest_df, on=['Employee_ID', 'Year', 'Month'], how='left').fillna(0)

monthly_summary['Monthly_Savings'] = monthly_summary['Monthly_Income'] - monthly_summary['Monthly_Expense']
monthly_summary['Savings_Percentage'] = (monthly_summary['Monthly_Savings'] / monthly_summary['Monthly_Income']) * 100
monthly_summary['Savings_Percentage'] = monthly_summary['Savings_Percentage'].fillna(0)

# Merge back to main df
df = pd.merge(df, monthly_summary[['Employee_ID', 'Year', 'Month', 'Monthly_Expense', 'Monthly_Savings', 'Savings_Percentage']], 
              on=['Employee_ID', 'Year', 'Month'], how='left')

print("Monthly Aggregations added successfully!")
df[['Employee_ID', 'Date', 'Transaction_Type', 'Category', 'Amount', 'Monthly_Savings', 'Savings_Percentage']].head(8)

In [7]:
# 4. Synthesizing the ground truth is_leakage target variable using rule-based scoring
# Ground truth logic details:
# - If Expense_Type == 'Want' (+2)
# - If Weekend_Flag == 'Yes' AND Expense_Type == 'Want' (+2)
# - If Subscription == 'Yes' AND Expense_Type == 'Want' (+2)
# - If Amount < 300 AND Expense_Type == 'Want' (+1) (micro spend want)
# - If Salary_Week == 'Yes' AND Expense_Type == 'Want' (+1) (post salary week splurge)
# - If Payment_Method in ['Credit Card', 'UPI'] AND Expense_Type == 'Want' (+1)

def calculate_leakage_score(row):
    if row['Transaction_Type'] != 'Expense':
        return 0
    score = 0
    if row['Expense_Type'] == 'Want':
        score += 2
        if row['Weekend_Flag'] == 'Yes':
            score += 2
        if row['Subscription'] == 'Yes':
            score += 2
        if row['Amount'] < 300:
            score += 1
        if row['Salary_Week'] == 'Yes':
            score += 1
        if row['Payment_Method'] in ['Credit Card', 'UPI']:
            score += 1
    return score

df['leakage_score'] = df.apply(calculate_leakage_score, axis=1)
df['is_leakage'] = df['leakage_score'].apply(lambda x: 1 if x >= 5 else 0)

print("Ground truth labels generated!")
print(df['is_leakage'].value_counts())
df[df['Transaction_Type']=='Expense'][['Category', 'Amount', 'Weekend_Flag', 'Salary_Week', 'leakage_score', 'is_leakage']].head(10)

## 5. Exploratory Data Analysis (EDA)

In this section, we analyze the transaction behavior of our simulated corporate employees to extract key financial trends.

In [8]:
# 1. Monthly Spending & Savings Trends
monthly_spend = df[df['Transaction_Type']=='Expense'].groupby(['Year', 'Month'])['Amount'].sum().reset_index()
monthly_spend['Date_Str'] = monthly_spend.apply(lambda r: f"{int(r.Year)}-{int(r.Month):02d}", axis=1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.lineplot(data=monthly_spend, x='Date_Str', y='Amount', marker='o', color='#3f51b5')
plt.title('Monthly Expense Trend', fontweight='bold')
plt.xticks(rotation=45)

monthly_savings = df.groupby(['Year', 'Month'])['Monthly_Savings'].mean().reset_index()
monthly_savings['Date_Str'] = monthly_savings.apply(lambda r: f"{int(r.Year)}-{int(r.Month):02d}", axis=1)

plt.subplot(1, 2, 2)
sns.lineplot(data=monthly_savings, x='Date_Str', y='Monthly_Savings', marker='s', color='#2e7d32')
plt.title('Average Savings Trend per Professional', fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [9]:
# 2. Need vs Want Spend Analysis
need_want_spend = df[df['Transaction_Type']=='Expense'].groupby('Expense_Type')['Amount'].sum().reset_index()
plt.figure(figsize=(6, 6))
plt.pie(need_want_spend['Amount'], labels=need_want_spend['Expense_Type'], autopct='%1.1f%%', startangle=90, colors=['#1565c0', '#e53935'])
plt.title('Expense Breakdown: Needs vs Wants', fontweight='bold')
plt.show()

In [10]:
# 3. Weekend vs Weekday Category Spend
weekend_spend = df[df['Transaction_Type']=='Expense'].groupby(['Weekend_Flag', 'Category'])['Amount'].mean().reset_index()
sns.barplot(data=weekend_spend, x='Category', y='Amount', hue='Weekend_Flag', palette='muted')
plt.title('Average Category Spend: Weekdays vs Weekends', fontweight='bold')
plt.xticks(rotation=45)
plt.ylabel('Average Amount (₹)')
plt.show()

In [11]:
# 4. Payment Method & Work Mode Impact
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
pm_counts = df[df['Transaction_Type']=='Expense']['Payment_Method'].value_counts().reset_index()
sns.barplot(data=pm_counts, x='Payment_Method', y='count', palette='viridis')
plt.title('Payment Method Distribution (Expense Count)', fontweight='bold')

plt.subplot(1, 2, 2)
work_mode_spend = df[df['Transaction_Type']=='Expense'].groupby('Work_Mode')['Amount'].mean().reset_index()
sns.barplot(data=work_mode_spend, x='Work_Mode', y='Amount', palette='rocket')
plt.title('Average Transaction by Work Mode', fontweight='bold')
plt.tight_layout()
plt.show()

In [12]:
# 5. Savings Percentage across Salary Bands and Top Leakage Categories
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
emp_dedup = df.drop_duplicates(subset=['Employee_ID'])
sns.boxplot(data=emp_dedup, x='Salary_Band', y='Savings_Percentage', palette='Set2')
plt.title('Savings Percentage by Salary Band', fontweight='bold')
plt.xticks(rotation=15)

plt.subplot(1, 2, 2)
leakage_by_cat = df[df['is_leakage']==1].groupby('Category')['Amount'].sum().sort_values(ascending=False).reset_index()
sns.barplot(data=leakage_by_cat, x='Amount', y='Category', palette='Reds_r')
plt.title('Top Leakage Categories by Total Spend', fontweight='bold')
plt.xlabel('Total Leakage Spend (₹)')
plt.tight_layout()
plt.show()

## 6. Machine Learning Model Training (Supporting Business Decisions)

We implement binary classification models to predict whether a transaction represents potential leakage. 
**CRITICAL DESIGN DETAIL:** To prevent target leakage and build a technically correct model, we completely drop `leakage_score` and `Expense_Type` before model training. The models must learn patterns solely from context: amount, merchant, payment channel, weekend flag, salary week, etc.

In [13]:
# Prepare features and target (expenses only)
ml_df = df[df['Transaction_Type'] == 'Expense'].copy()

# Drop metadata, targets, and leakage variables
features_to_drop = [
    'Transaction_ID', 'Employee_ID', 'Date', 'Transaction_Type', 
    'Merchant', 'leakage_score', 'is_leakage', 'leakage_reason',
    'Year', 'Month', 'Day_of_Week', 'Monthly_Expense', 'Monthly_Savings', 
    'Savings_Percentage', 'Expense_Type'
]

X = ml_df.drop(columns=features_to_drop, errors='ignore')
y = ml_df['is_leakage']

# Stratified Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [14]:
# Preprocessing pipeline
categorical_cols = ['Salary_Band', 'Work_Mode', 'Category', 'Payment_Method', 'Weekend_Flag', 'Salary_Week', 'Subscription', 'Time_of_Day']
numeric_cols = ['Monthly_Salary', 'Amount']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# 1. Logistic Regression Model
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

print("Training Logistic Regression Classifier...")
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression Evaluation:")
print(classification_report(y_test, lr_preds))

In [15]:
# 2. Random Forest Classifier Model
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', max_depth=12))
])

In [16]:
print("Training Random Forest Classifier...")
rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest Evaluation:")
print(classification_report(y_test, rf_preds))

In [17]:
# Plot Confusion Matrix for Random Forest
cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Leakage'], yticklabels=['Normal', 'Leakage'])
plt.title('Random Forest Confusion Matrix', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [18]:
# Plot Feature Importances (Random Forest)
rf_model = rf_pipeline.named_steps['classifier']
feature_names = numeric_cols + list(rf_pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols))
importances = rf_model.feature_importances_

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False).head(15)

sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
plt.show()

## 7. Actionable Recommendations & Predictions Export

In [19]:
# Apply predictions back to the full dataset
full_X = df[df['Transaction_Type'] == 'Expense'].drop(columns=features_to_drop, errors='ignore')
df.loc[df['Transaction_Type'] == 'Expense', 'predicted_leakage'] = rf_pipeline.predict(full_X)
df.loc[df['Transaction_Type'] == 'Expense', 'predicted_leakage_prob'] = rf_pipeline.predict_proba(full_X)[:, 1]

df['predicted_leakage'] = df['predicted_leakage'].fillna(0).astype(int)
df['predicted_leakage_prob'] = df['predicted_leakage_prob'].fillna(0)

processed_path = "../data/processed/cleaned_transaction_data.csv"
df.to_csv(processed_path, index=False)
print(f"Successfully saved processed predictions to {processed_path}!")

In [20]:
print("Sample of transactions predicted as leakage with reasons:")
leakages = df[df['predicted_leakage'] == 1]
leakages[['Category', 'Amount', 'Payment_Method', 'Weekend_Flag', 'Salary_Week', 'leakage_reason']].head(10)